# Chocolate price guesses: packaging, store and brand

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Factorial Repeated Measures Analysis of Variance** · a worked example from the StatNex teaching guide

**The story.** Eight shoppers guessed the price of the same box of chocolates shown in plain or premium packaging and pictured in a discount store, a supermarket or a department store. Four shoppers knew the brand and four did not, and every shopper saw all six versions.

**The question.** How much do packaging and store setting raise the price people expect, and does knowing the brand protect against these cues?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `05_anova/factorial_anova_repeated.csv` (48 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Knows the brand,Packaging,Store,Price guess ($),Shopper
No,1.0,1.0,3.5,1.0
No,1.0,1.0,4.3,2.0
No,1.0,1.0,5.4,3.0
No,1.0,1.0,4.0,4.0
No,1.0,2.0,4.0,1.0
No,1.0,2.0,5.7,2.0
No,1.0,2.0,5.7,3.0
No,1.0,2.0,5.7,4.0
No,1.0,3.0,6.2,1.0
No,1.0,3.0,7.1,2.0
No,1.0,3.0,7.7,3.0
No,1.0,3.0,7.1,4.0
No,2.0,1.0,5.6,1.0
No,2.0,1.0,5.7,2.0
No,2.0,1.0,6.9,3.0
No,2.0,1.0,6.5,4.0
No,2.0,2.0,6.6,1.0
No,2.0,2.0,7.3,2.0
No,2.0,2.0,8.6,3.0
No,2.0,2.0,7.3,4.0
No,2.0,3.0,9.1,1.0
No,2.0,3.0,8.6,2.0
No,2.0,3.0,9.3,3.0
No,2.0,3.0,9.3,4.0
Yes,1.0,1.0,5.7,5.0
Yes,1.0,1.0,5.1,6.0
Yes,1.0,1.0,5.4,7.0
Yes,1.0,1.0,5.6,8.0
Yes,1.0,2.0,5.7,5.0
Yes,1.0,2.0,5.6,6.0
Yes,1.0,2.0,5.8,7.0
Yes,1.0,2.0,6.4,8.0
Yes,1.0,3.0,6.0,5.0
Yes,1.0,3.0,6.1,6.0
Yes,1.0,3.0,6.0,7.0
Yes,1.0,3.0,7.6,8.0
Yes,2.0,1.0,6.6,5.0
Yes,2.0,1.0,5.9,6.0
Yes,2.0,1.0,5.6,7.0
Yes,2.0,1.0,6.7,8.0
Yes,2.0,2.0,6.9,5.0
Yes,2.0,2.0,6.8,6.0
Yes,2.0,2.0,7.0,7.0
Yes,2.0,2.0,7.9,8.0
Yes,2.0,3.0,7.5,5.0
Yes,2.0,3.0,6.7,6.0
Yes,2.0,3.0,7.1,7.0
Yes,2.0,3.0,7.6,8.0

These cells save `eta_squared_interval.py`, `sphericity.py`: steps that no library does, written out. The analysis imports them.

In [ ]:
%%writefile eta_squared_interval.py
"""The confidence interval for partial eta squared, from the noncentral F distribution.

SciPy has the noncentral F (stats.ncf) but no function that inverts it for an interval,
so the inversion is written out (Steiger 2004). The interval is found for the
noncentrality λ and converted to partial eta squared as λ / (λ + df1 + df2 + 1).
"""

import numpy as np
from scipy import optimize, stats


def _limit(f, df1, df2, q):
    """The λ ≥ 0 under which the observed F sits at the q quantile of F(df1, df2, λ).

    0 when even λ = 0 cannot put it there: an F too small for the interval to leave 0.
    The cdf at F falls as λ rises, so the bracket opens outwards from F·df1, doubling.
    """
    def gap(lam):
        cdf = stats.f.cdf(f, df1, df2) if lam == 0 else stats.ncf.cdf(f, df1, df2, lam)
        return cdf - q

    if gap(0.0) <= 0:
        return 0.0
    start = max(f * df1, 1.0)
    step = 1.0
    while gap(max(start - step, 0.0)) <= 0:
        step *= 2
    low = max(start - step, 0.0)
    step = 1.0
    while gap(start + step) >= 0:
        step *= 2
    return optimize.brentq(gap, low, start + step, xtol=1e-14,
                           rtol=4 * np.finfo(float).eps, maxiter=500)


def partial_eta_squared_interval(f, df1, df2, level):
    """(lower, upper): the level interval for partial eta squared (Steiger 2004)."""
    total = df1 + df2 + 1
    lower = _limit(f, df1, df2, (1 + level) / 2)
    upper = _limit(f, df1, df2, (1 - level) / 2)
    return lower / (lower + total), upper / (upper + total)

In [ ]:
%%writefile sphericity.py
"""Mauchly's test of sphericity and the epsilon corrections for repeated measures.

statsmodels' AnovaRM prints only the uncorrected F test and SciPy has neither, so both
are written out. Each takes the covariance matrix of one within-subject effect's
orthonormal contrast scores -- the residuals of the between-subject model when there is
one -- which is all that sphericity is about.
"""

import numpy as np
from scipy import stats


def mauchly(covariance, df):
    """Mauchly's W, its chi-square approximation, the chi-square df and P.

    ``df`` is the error df of the covariance matrix (subjects minus groups). W is 1 when
    the contrasts are uncorrelated with equal variances, which is what sphericity means.
    With one contrast -- a two-level factor -- there is nothing to test, so W = 1 and P
    is undefined.
    """
    p = covariance.shape[0]
    if p < 2:
        return 1.0, 0.0, 0, np.nan
    w = np.linalg.det(covariance) / (np.trace(covariance) / p) ** p
    correction = 1 - (2 * p**2 + p + 2) / (6 * p * df)
    chi_square = -df * correction * np.log(w)
    chi_df = p * (p + 1) // 2 - 1
    return w, chi_square, chi_df, stats.chi2.sf(chi_square, chi_df)


def epsilons(covariance, n, groups):
    """(lower bound, Greenhouse-Geisser, Huynh-Feldt-Lecoutre, Chi-Muller) for one effect.

    Epsilon is 1 under sphericity and 1/p at worst. Huynh-Feldt's estimate uses
    Lecoutre's (1991) correction, (N - g + 1) in the numerator, and can exceed 1; the
    table caps it at 1 when it multiplies the degrees of freedom. Chi-Muller shrinks it,
    since it runs high in small samples, by a multiple fixed by the design alone,
    (m - 1)(m - 2) / m^2 with m = p (N - g - 1), and floors it at the lower bound
    (Chi et al. 2012; SAS's UEPSDEF=CM).
    """
    p = covariance.shape[0]
    lower = 1 / p
    gg = np.trace(covariance) ** 2 / (p * np.trace(covariance @ covariance))
    gg = min(max(gg, lower), 1.0)  # rounding can step a hair outside [1/p, 1]
    df = n - groups
    hfl = ((df + 1) * p * gg - 2) / (p * (df - p * gg))
    m = p * (df - 1)
    cm = max(hfl * (m - 1) * (m - 2) / m**2, lower) if m > 0 else lower
    return lower, gg, hfl, cm

## Libraries

In [ ]:
from functools import reduce
from itertools import combinations

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from scipy.linalg import helmert
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multitest import multipletests
from sphericity import epsilons, mauchly

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per measurement: the response, the subject, the within-subject factors (every
# subject meets every combination) and the between-subject factors (one level per
# subject). Rows with a blank are dropped; every subject must still have exactly one
# measurement in every within-subject cell. Levels keep the order they appear in.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "Price guess ($)"
subject = "Shopper"
within = ["Packaging", "Store"]
between = ["Knows the brand"]
data[response] = pd.to_numeric(data[response], errors="coerce")
data = data.dropna(subset=[response, subject, *within, *between]).reset_index(drop=True)
for name in [subject, *within, *between]:
    data[name] = pd.Categorical(data[name], categories=data[name].unique())
levels = {name: list(data[name].cat.categories) for name in within}

# Subjects down the rows, within-subject cells across the columns (the first factor
# changing slowest). pivot refuses a subject measured twice in one cell. With one
# within-subject factor pivot gives plain columns, which are made a one-level MultiIndex
# like the cells, or the reindex would match none of them.
cells = pd.MultiIndex.from_product([levels[name] for name in within])
wide = data.pivot(index=subject, columns=within, values=response)
if not isinstance(wide.columns, pd.MultiIndex):
    wide.columns = pd.MultiIndex.from_arrays([wide.columns.astype(object)])
wide = wide.reindex(columns=cells)
if wide.isna().any().any():
    raise ValueError("every subject needs a measurement in every within-subject cell")
y = wide.to_numpy(float)
n, K = y.shape  # subjects, within-subject cells
subjects = data.groupby(subject, observed=True)[between].first().loc[wide.index]

## Descriptive statistics

In [ ]:
# Every factor and every combination of factors.
factors = [*between, *within]
pieces = []
for size in range(1, len(factors) + 1):
    for chosen in combinations(factors, size):
        summary = data.groupby(list(chosen), observed=True)[response].agg(
            n="count", mean="mean", SD="std")
        summary.index = [" * ".join(map(str, np.atleast_1d(key))) for key in summary.index]
        summary.insert(0, "model", " * ".join(chosen))
        pieces.append(summary)
descriptive = pd.concat(pieces)
descriptive["SE"] = descriptive["SD"] / np.sqrt(descriptive["n"])
descriptive = descriptive.assign(**{
    "mean - SD": descriptive["mean"] - descriptive["SD"],
    "mean + SD": descriptive["mean"] + descriptive["SD"],
    "mean - SE": descriptive["mean"] - descriptive["SE"],
    "mean + SE": descriptive["mean"] + descriptive["SE"],
})[["model", "n", "mean", "SD", "mean - SD", "mean + SD", "SE", "mean - SE", "mean + SE"]]
print(descriptive.to_string())

## Test for homogeneity of variance

In [ ]:
# Between the groups, separately for each within-subject cell (only when there are
# groups). Bartlett skips a group with one subject or no spread; Levene's original
# form centres on each group's mean.
if between:
    members = subjects.reset_index(drop=True).groupby(between, sort=False, observed=True).indices
    rows = []
    for j, cell in enumerate(cells):
        samples = [y[index, j] for index in members.values()]
        usable = [s for s in samples if len(s) > 1 and s.var(ddof=1) > 0]
        bartlett = stats.bartlett(*usable)
        levene = stats.levene(*samples, center="mean")
        rows.append([" * ".join(map(str, cell)), bartlett.statistic, len(usable) - 1,
                     bartlett.pvalue, levene.statistic, len(samples) - 1, n - len(samples),
                     levene.pvalue])
    homogeneity = pd.DataFrame(rows, columns=["cell", "Bartlett chi-square", "df", "P",
                                              "Levene F", "df1", "df2", "P "])
    print()
    print(homogeneity.to_string(index=False))

## Between subjects

In [ ]:
# Each subject's mean over the within-subject cells, compared across the groups:
# Type III sums of squares with sum-to-zero coding, as the app uses.
terms = [" * ".join(c) for size in range(1, len(between) + 1) for c in combinations(between, size)]
formula = " * ".join(f"C(Q({name!r}), Sum)" for name in between) or "1"
# The response goes into the same frame, under a name no group column has: a formula
# reads a name from the frame first, so a group called "score" would stand in for it.
score = "score"
while score in between:
    score += "_"
subjects[score] = y.mean(axis=1)
between_table = anova_lm(smf.ols(f"{score} ~ {formula}", data=subjects).fit(), typ=3)
between_table.index = ["Intercept", *terms, "Residual"]
between_table["sum_sq"] *= K  # a subject mean stands for K measurements

## Within subjects

In [ ]:
# Each within-subject effect is a set of contrasts among the cells: orthonormal
# contrasts (scaled Helmert) on the factors it names, averages over the others. The
# contrast scores are regressed on the group model; the intercept is the effect, each
# group term its interaction with the effect, the residual its own error term.
def contrasts(effect):
    blocks = [helmert(len(levels[name])).T if name in effect
              else np.full((len(levels[name]), 1), 1 / np.sqrt(len(levels[name])))
              for name in within]
    return reduce(np.kron, blocks)


effects = [c for size in range(1, len(within) + 1) for c in combinations(within, size)]
results = {}
for effect in effects:
    scores = y @ contrasts(effect)
    fits = [smf.ols(f"{score} ~ {formula}", data=subjects.assign(**{score: scores[:, j]})).fit()
            for j in range(scores.shape[1])]
    table = sum(anova_lm(fit, typ=3)[["sum_sq"]] for fit in fits)
    table["df"] = anova_lm(fits[0], typ=3)["df"] * scores.shape[1]
    residuals = np.column_stack([fit.resid for fit in fits])
    df_resid = fits[0].df_resid  # subjects minus groups
    covariance = residuals.T @ residuals / df_resid
    results[" * ".join(effect)] = (table, covariance, df_resid)

## Mauchly's test of sphericity

In [ ]:
# One test per within-subject effect, on the covariance of its contrasts' residuals.
# A two-level effect has one contrast and nothing to test (W = 1, no P).
rows = []
for name, (table, covariance, df_resid) in results.items():
    rows.append([*mauchly(covariance, df_resid), *epsilons(covariance, n, groups=n - df_resid)])
sphericity = pd.DataFrame(rows, index=list(results),
                          columns=["W", "chi-square", "df", "P", "lower bound",
                                   "Greenhouse-Geisser", "Huynh-Feldt-Lecoutre", "Chi-Muller"])
print()
print(sphericity.to_string())

## Analysis of variance table

In [ ]:
# Within-subject rows are tested against their own error term, five ways: each
# correction multiplies both degrees of freedom by its epsilon (Huynh-Feldt capped at
# 1). Between-subject rows are tested against how much subjects differ in each group.
rows = []
for name, (table, covariance, df_resid) in results.items():
    lower, gg, hfl, cm = sphericity.loc[name, ["lower bound", "Greenhouse-Geisser",
                                               "Huynh-Feldt-Lecoutre", "Chi-Muller"]]
    corrections = {"Sphericity assumed": 1.0, "Greenhouse-Geisser": gg,
                   "Huynh-Feldt-Lecoutre": min(hfl, 1.0), "Chi-Muller": min(cm, 1.0),
                   "Lower bound": lower}
    ss_error, df_error = table.loc["Residual", "sum_sq"], table.loc["Residual", "df"]
    sources = [name, *[f"{term} * {name}" for term in terms]]
    for source, ss, df in zip(sources, table["sum_sq"].iloc[:-1], table["df"].iloc[:-1]):
        f_value = (ss / df) / (ss_error / df_error)
        for correction, epsilon in corrections.items():
            df1, df2 = df * epsilon, df_error * epsilon
            rows.append([source, correction, ss, df1, ss / df1, f_value,
                         stats.f.sf(f_value, df1, df2)])
    for correction, epsilon in corrections.items():
        df2 = df_error * epsilon
        rows.append([f"Error ({name})", correction, ss_error, df2, ss_error / df2,
                     np.nan, np.nan])
ss_subject = between_table.loc["Residual", "sum_sq"]
df_subject = between_table.loc["Residual", "df"]
for source in terms:
    ss, df = between_table.loc[source, "sum_sq"], between_table.loc[source, "df"]
    f_value = (ss / df) / (ss_subject / df_subject)
    rows.append([source, "", ss, df, ss / df, f_value, stats.f.sf(f_value, df, df_subject)])
rows.append([f"{subject} (error)", "", ss_subject, df_subject, ss_subject / df_subject,
             np.nan, np.nan])
rows.append(["Total", "", ((y - y.mean()) ** 2).sum(), n * K - 1, np.nan, np.nan, np.nan])
anova = pd.DataFrame(rows, columns=["source", "correction", "SS", "df", "MS", "F", "P"])
print()
print(anova.to_string(index=False))

## Effect sizes

In [ ]:
# Partial eta² sets each effect against its own error term: SS / (SS + SS error), a
# within-subject effect's own residual for it and its interactions with the groups, the
# subjects for a group effect. It carries the 95% interval that inverts the noncentral
# F at the sphericity-assumed F (Steiger 2004), written out in eta_squared_interval.py.
# Generalised eta² sets every effect against all the error terms at once -- the
# subjects and every within-subject effect's error -- so it compares across designs
# (Olejnik & Algina 2003, every factor treated as manipulated).
from eta_squared_interval import partial_eta_squared_interval


def size(value):
    """Cohen's (1988) reading of an eta²: 0.01 small, 0.06 medium, 0.14 large."""
    return ["negligible", "small", "medium", "large"][sum(value >= s for s in (0.01, 0.06, 0.14))]


strata = []
for name, (table, _, _) in results.items():
    error, error_df = table.loc["Residual", "sum_sq"], table.loc["Residual", "df"]
    sources = [name, *[f"{term} * {name}" for term in terms]]
    strata += [(source, ss, df, error, error_df) for source, ss, df
               in zip(sources, table["sum_sq"].iloc[:-1], table["df"].iloc[:-1])]
strata += [(source, between_table.loc[source, "sum_sq"], between_table.loc[source, "df"],
            ss_subject, df_subject) for source in terms]
all_errors = ss_subject + sum(table.loc["Residual", "sum_sq"] for table, _, _ in results.values())
rows = []
for source, ss, df, error, error_df in strata:
    lower, upper = partial_eta_squared_interval((ss / df) / (error / error_df), df, error_df, 0.95)
    rows.append([source, "partial eta²", ss / (ss + error), lower, upper])
    rows.append([source, "generalised eta²", ss / (ss + all_errors), np.nan, np.nan])
effect_sizes = pd.DataFrame(rows, columns=["effect", "measure", "value", "lower", "upper"])
effect_sizes["size"] = effect_sizes["value"].map(size)
print()
print(effect_sizes.to_string(index=False))

## Multiple comparisons

In [ ]:
# For each within-subject factor, every pair of its levels (averaged over the other
# factors), with a t statistic on that factor's error term, then each P value adjusted
# for the number of pairs by the chosen procedures.
# Any of "Bonferroni", "Sidak" and "Holm", in the order to print them; [] for none.
procedures = []
rows = []
for position, name in enumerate(within):
    table = results[name][0]
    ms_error = table.loc["Residual", "sum_sq"] / table.loc["Residual", "df"]
    df_error = table.loc["Residual", "df"]
    in_level = [cells.get_level_values(position) == level for level in levels[name]]
    means = [y[:, mask].mean() for mask in in_level]
    se_difference = np.sqrt(2 * ms_error / (n * K / len(levels[name])))
    pairs = list(combinations(range(len(levels[name])), 2))
    difference = np.array([means[i] - means[j] for i, j in pairs])
    t_value = difference / se_difference
    p_raw = 2 * stats.t.sf(np.abs(t_value), df_error)
    for procedure in procedures:
        p_adjusted = multipletests(p_raw, method=procedure.lower())[1]
        for (i, j), d, t, p in zip(pairs, difference, t_value, p_adjusted):
            rows.append([name, procedure, levels[name][i], levels[name][j], means[i], means[j],
                         d, se_difference, t, p])
comparisons = pd.DataFrame(rows, columns=["factor", "method", "level 1", "level 2", "mean 1",
                                          "mean 2", "difference", "SE", "t", "P"])
print()
print(comparisons.to_string(index=False))

## What the result says

Every shopper saw all six versions, so packaging and store are compared inside each shopper. F compares an effect with the noise, and the P value is the chance of an effect at least this large if there were none.

Packaging has F = 168.7 and Store has F = 120.1, both with P values far below 0.01. The mean guess rises from \$5.73 for plain to \$7.21 for premium packaging, and from \$5.53 in the discount store to \$7.44 in the department store.

An interaction means one factor's effect depends on another. Knows the brand * Packaging has P = 0.003: with premium packaging, shoppers who do not know the brand go from \$5.53 to \$7.57, and those who know it only from \$5.92 to \$6.86. Both cues raise the expected price, and knowing the brand weakens them.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.